MAT-4 FINAL PROJECT

GROUP **5**

**Fast Algorithms for Solving Linear Systems of Equations with Toeplitz or Hankel Coefficient Matrices**

FFT Solver

In [9]:
import numpy as np
from numpy.fft import fft, ifft
from scipy.linalg import toeplitz

def generate_exp_decay_toeplitz(n, alpha=0.5):
    c = alpha ** np.arange(n)
    return toeplitz(c)

def generate_system_from_toeplitz(T, seed=None):
    n = T.shape[0]
    if seed is not None:
        np.random.seed(seed)
    x_true = np.random.rand(n)
    b = T @ x_true
    return x_true, b

def solve_toeplitz_fft(c, b):
    n = len(c)
    circ = np.concatenate([c, c[-2:0:-1]])
    """
    print("Length of c:", len(c))
    print("Length of circ:", len(circ))
    print("First 10 elements of circ:", circ[:10])
    print("Last 10 elements of circ:", circ[-10:])
    """

    fft_circ = fft(circ)
    b_ext = np.zeros_like(circ)
    b_ext[:n] = b
    fft_b = fft(b_ext)
    threshold = 1e-15
    denom = np.where(np.abs(fft_circ) < threshold, threshold, fft_circ)

    fft_x = fft_b / denom
    x_ext = ifft(fft_x).real
    return x_ext[:n]

def test_fft_solver(n=32):
    alpha=0.1
    T = generate_exp_decay_toeplitz(n, alpha)
    x_true, b = generate_system_from_toeplitz(T, seed=42)
    c = T[:, 0]

    x_fft = solve_toeplitz_fft(c, b)
    abs_error = np.abs(x_fft - x_true)
    rel_error = abs_error / (np.abs(x_true) + 1e-12)
    l2_error = np.linalg.norm(x_fft - x_true)

    print("\nFFT-based solver test")
    print(f"Matrix size (n):     {n}")
    print(f"Max absolute error:  {np.max(abs_error)}")
    print(f"Max relative error:  {np.max(rel_error)}")
    print(f"L2 norm of error:    {l2_error}")

test_fft_solver(n=512)


FFT-based solver test
Matrix size (n):     512
Max absolute error:  0.004928112661380768
Max relative error:  0.012879025452326017
L2 norm of error:    0.006895976304724552


Levinsons

In [11]:
import numpy as np

def solve_toeplitz_levinson(c_or_cr, b, check_finite=True):
    """
    Solve a Toeplitz system T x = b using Levinson–Durbin recursion.

    Parameters
    ----------
    c_or_cr : array_like or tuple of (array_like, array_like)
        - If array_like: interpreted as `c`, the first column of T.
          Then `r = conjugate(c)` is assumed (Hermitian Toeplitz).
        - If tuple (c, r): `c` is the first column, `r` is the first row.
          In that case, r[0] is ignored, and T[0,1:] = r[1:].
    b : array_like, shape (n,) or (n, k)
        Right‐hand side(s). If `b` is 2D, each column is treated as a separate RHS.
    check_finite : bool, optional
        If True, checks that inputs contain only finite numbers.

    Returns
    -------
    x : ndarray, shape (n,) or (n, k)
        The solution(s) to T x = b.  If `b` was (n, k), returns (n, k).
    """
    # 1) Parse inputs exactly as scipy.linalg.solve_toeplitz does :contentReference[oaicite:1]{index=1}
    if check_finite:
        # we'll rely on NumPy for a quick finite‐check (raise if any inf/NaN)
        c_or_cr = np.asarray(c_or_cr, dtype=np.complex128)
        b = np.asarray(b, dtype=np.complex128)

    # Unpack c and r
    if isinstance(c_or_cr, tuple):
        c = np.asarray(c_or_cr[0], dtype=np.complex128).ravel()
        r = np.asarray(c_or_cr[1], dtype=np.complex128).ravel()
        if c.shape != r.shape:
            raise ValueError("First column and row must be the same length")
    else:
        c = np.asarray(c_or_cr, dtype=np.complex128).ravel()
        # Hermitian Toeplitz ⇒ first row = conj(c)
        r = np.conjugate(c)

    n = c.size
    if b.ndim == 1:
        b = b.reshape(n, 1)
    elif b.ndim == 2:
        if b.shape[0] != n:
            raise ValueError("Dimension mismatch: b has length %d but c has length %d" % (b.shape[0], n))
    else:
        raise ValueError("b must be 1‐ or 2‐dimensional")

    # 2) Solve each column of b with Levinson–Durbin
    # Preallocate output
    x = np.zeros_like(b, dtype=np.complex128)

    # The core Levinson–Durbin routine for one RHS (1D arrays)
    def _levinson_durbin(c_col, r_row, b_col):
        """
        Solve T x = b_col for x, where T is the n×n Toeplitz with
        first column c_col and first row r_row, using Levinson–Durbin.
        """
        # Ensure lengths
        n = c_col.size
        # Allocate
        x = np.zeros(n, dtype=np.complex128)
        # Temporary arrays for recursion
        g = np.zeros(n, dtype=np.complex128)  # reflection coefficients
        phi = np.zeros(n, dtype=np.complex128)  # “backward” recursions

        #  --- Step k = 0  (initialize) ---
        # T[0,0] = c_col[0] = r_row[0]; must be nonzero
        if c_col[0] == 0:
            raise np.linalg.LinAlgError("Singular principal minor (c[0] = 0)")

        # x[0] = b[0] / c[0]
        x[0] = b_col[0] / c_col[0]
        # φ[0] = -r[1] / c[0]   (reflection coefficient for k=1)
        if n > 1:
            phi[0] = -r_row[1] / c_col[0]
        beta = c_col[0]  # β₀ = c[0]

        #  --- Recursion for k = 1 to n-1 ---
        for k in range(1, n):
            # 1) Compute the “forward” prediction error:
            #    λ = (b[k] - ∑_{j=0..k-1} c[j+1] * x[k-1-j]) / β_{k-1}
            # Note: c[j+1] = T[k, k-j-1], and x[k-1-j] = previous solution part reversed
            # We can do dot(c_col[1:k+1], x[k-1::-1]) for ∑ c[j+1]*x[k-1-j].
            num = b_col[k] - np.dot(c_col[1:k+1], x[:k][::-1])
            lam = num / beta

            # Update x[0..k-1] ← x[0..k-1] + λ * φ[k-1..0]
            x[:k] += lam * phi[k-1::-1]
            # Set x[k] = λ
            x[k] = lam

            if k == n-1:
                break

            # 2) Compute next reflection coefficient μ:
            #    μ = - (r[k+1] + ∑_{j=0..k-1} c[j+1] * φ[k-1-j]) / β_{k-1}
            num2 = r_row[k+1] + np.dot(c_col[1:k+1], phi[k-1::-1])
            mu = -num2 / beta

            # Update φ[0..k-1] ← φ[0..k-1] + μ * φ[k-1..0]
            phi[:k] += mu * phi[k-1::-1]
            phi[k] = mu

            # 3) Update β ← β * (1 - |μ|^2)
            beta = beta * (1 - mu * np.conjugate(mu))
            if beta == 0:
                raise np.linalg.LinAlgError("Singular principal minor at step k = %d" % k)

        return x


    # Now apply _levinson_durbin to each column of b
    for col_idx in range(b.shape[1]):
        x[:, col_idx] = _levinson_durbin(c, r, b[:, col_idx])

    # If original b was 1D, return a 1D result
    if x.shape[1] == 1:
        return x.ravel()
    else:
        return x


In [13]:
import numpy as np
import scipy.linalg

# 1) Create a symmetric Toeplitz matrix T via its first column c
n = 5
alpha = 0.3
c = alpha ** np.arange(n)           # [1, 0.3, 0.09, 0.027, 0.0081]
# If you pass just c, r=conjugate(c) is assumed → Hermitian Toeplitz

# 2) Pick a "true" solution vector x_true, compute b = T @ x_true
np.random.seed(0)
x_true = np.random.rand(n)
# Form the full Toeplitz to verify
from scipy.linalg import toeplitz
T = toeplitz(c, c)  # since r = c here (real-valued, symmetric)
b = T @ x_true

# 3) Solve via our Levinson-based routine
x_levinson = solve_toeplitz_levinson(c, b)
print(x_true)
x_true = scipy.linalg.solve_toeplitz(c, b)

# 4) Compare
print("True x:",      x_true)
print("Levinson x:",  x_levinson)
print("Max abs err:", np.max(np.abs(x_levinson - x_true)))


[0.5488135  0.71518937 0.60276338 0.54488318 0.4236548 ]
True x: [0.5488135  0.71518937 0.60276338 0.54488318 0.4236548 ]
Levinson x: [0.57463891+0.j 0.64377199+0.j 0.56589652+0.j 0.5111229 +0.j
 0.40258714+0.j]
Max abs err: 0.0714173790085244


Library used (for levinsons) : https://github.com/scipy/scipy/blob/main/scipy/linalg/_basic.py#L898